# Análisis V2 - Regresión Lineal Simple: Producción de Arroz

## 📌 Contexto

Este notebook documenta la **segunda versión (V2)** del modelo de Regresión Lineal Simple para predecir la **Producción (qq)** de arroz a partir de la **Superficie (Ha)** cultivada.

### ¿Por qué una V2?

El modelo V1 presentaba un problema serio: aunque su R² era aceptable (0.766), su **MAPE era de 110.44%**, lo que significa que en promedio el modelo erraba por más del 100% del valor real. En la práctica, eso lo hacía inservible.

### Hipótesis de mejora

1. **Outliers**: existe un registro extremo (8,545 Ha → 804,641 qq) que sesga la pendiente.
2. **Asimetría**: la distribución de Superficie tiene cola larga a la derecha; una **transformación logarítmica** debería linealizar la relación.

## 1. Metodología V2

Se probaron **4 escenarios** cruzando dos decisiones:

| Dimensión | Opción 1 | Opción 2 |
|---|---|---|
| Datos | Originales (1375) | Filtrados (1373, SUPERFICIE < 3000 Ha) |
| Transformación | Sin log | Con log1p en X e Y |

**Escenarios:**
- **A)** Original (sin filtro, sin log)
- **B)** Filtrado (sin log)
- **C)** Original + Log
- **D)** Filtrado + Log

## 2. Resultados

Tabla comparativa obtenida al ejecutar `08_Regresion_Lineal_V2.py`:

| Modelo | R² | RMSE | MAE | MAPE (%) | MSE |
|---|---|---|---|---|---|
| A) Original | 0.7664 | 19427.76 | 2136.91 | **110.44** | 3.77e+08 |
| B) Filtrado | 0.4836 | 6264.99 | 1806.14 | 201.45 | 3.93e+07 |
| C) Original + Log | 0.4639 | 29430.61 | 2744.29 | 52.93 | 8.66e+08 |
| **D) Filtrado + Log** | **0.6898** | **4855.97** | **1334.91** | **49.35** | **2.36e+07** |

## 3. Interpretación

### 🔴 A) Original: el espejismo del R²
Aunque el R² (0.766) parece el segundo mejor, el **MAPE (110%)** delata que el modelo está dominado por el outlier gigante. Predice bien el punto extremo pero falla en el 99% restante.

### 🟡 B) Filtrado solo: empeora
Al filtrar sin log, el R² cae a 0.48 y el MAPE **sube a 201%**. Quitar el outlier destapó la dispersión real: hay parcelas con la misma superficie y producciones muy distintas. El logaritmo es necesario.

### 🔵 C) Original + Log: el log hace magia
Aplicando log, el MAPE cae de 110% a **52.93%** incluso con el outlier. La relación Superficie→Producción es **multiplicativa**, no aditiva.

### 🟢 D) Filtrado + Log: la mejor combinación 🏆
Al combinar **ambas mejoras**, obtenemos:
- Mejor R² entre los escenarios realistas: **0.6898**
- Mejor RMSE: **4855.97** (4x menor que A)
- Mejor MAE: **1334.91**
- Mejor MAPE: **49.35%** (bajó 61 puntos vs V1)

### Conclusión
> El logaritmo aporta **más que el filtrado por sí solo**. Filtrar ayuda únicamente cuando ya aplicamos log.

## 4. Visualización

![Regresión Lineal V2](../03_imagenes/24_regresion_lineal_v2.png)

**Observaciones de la gráfica:**
- A) y B) muestran el ajuste en escala original: la recta apenas captura la curvatura natural.
- C) y D) muestran el ajuste en escala logarítmica: la relación se vuelve claramente lineal.

## 5. Comparación con V1

| Métrica | V1 (Original) | V2 D (Mejor) | Mejora |
|---|---|---|---|
| R² | 0.7664 | 0.6898 | -0.077 |
| RMSE | 19427.76 | **4855.97** | **-75%** |
| MAE | 2136.91 | **1334.91** | **-38%** |
| MAPE | 110.44% | **49.35%** | **-61 pp** |

> Aunque el R² es ligeramente menor, **todas las métricas de error absoluto y relativo mejoran drásticamente**. En un caso de uso real, el V2-D es infinitamente más útil.

## 6. Conclusiones y próximos pasos

### ✅ Aprendizajes
1. El R² puede ser engañoso cuando hay outliers: **siempre revisar MAPE y MAE en conjunto**.
2. La relación Superficie→Producción es **multiplicativa** (log mejora mucho).
3. Filtrar outliers sin transformar **no ayuda** por sí solo.

### 🔜 Próximos pasos
1. Comparar con el modelo Polinomial V2 (`09_Analisis_Regresion_Polinomial_V2.ipynb`).
2. Explorar si agregar variables (campaña, departamento, rendimiento) reduce el MAPE < 30%.
3. Probar umbral de outliers alternativo (ej. 2000 Ha).